## <center>Table of content</center>
<br>
<center>
<div align="center">
<b>1. Overview
    <br><br>
2. Installation of HLS4ML
    <br><br>
3. Keras to HLS4ML example
    <br><br>
4. Compilation & build process
    <br><br>
5. Result of Design Flow
    <br><br>
    6. Conclusion</b>
    </div>
</center>

# <center>1. Overview</center>

## <center>Tools for ML-to-FPGA conversion</center>
<br>
<center>
<b>TVM/VTA
<br><br>
HADDOC2
<br><br>    
    HLS4ML</b>
</center>

## <center>HLS4ML</center>
<br>
<center>
    <a href=https://fastmachinelearning.org/hls4ml/CONCEPTS.html>Concept</a>
</center>

# <center>2. Installation of HLS4ML</center>
<br>
<center>Code, files and everything needed is available</center>
<br>
<center>on GitHub</center>
<br>
<center>https://github.com/fprotopapa/hls4ml-tutorial.git</center>

## <center>Requirements</center>

### <center>Operating System</center>
<br>
<center>First, we should be using the Ubuntu operating system.</center> 
<br>
<center>Useful installation guides for the operating system and other tools are listed in the repository.</center>
<br>
<center>Here we're using version 18.04.</center>

<center>To check your ubuntu version, run:</center>

In [ ]:
%%bash 
cat /etc/lsb-release

### <center>Then, we need Xilinx Vivado.</center>
<br>
<center>The tool is available on Xilinx website.</center> 
<br>
<center>Useful installation guides are again listed in the repository.</center>
<br>
<center>Here, the version used is 2019.2.</center>

### <center>Locating Vivado.</center>
<br>
<center>Vivado is located at</center>
<br>
<center><b>/tools/Xilinx/Vivado/2019.2/</b></center> 
<br>
<center>or sometimes under</center> 
<br>
<center><b>/opt/Xilinx/Vivado/2019.2/</b></center>
<br>
<center>this information will be needed later on.</center>

### <center>Let's install Python 3</center>
<br>
<center>and the <b>venv package</b></center>
<br>
<center>by running</center> 
<br>
<center><code>sudo apt install python3 python3-venv</code></center>
<br>
<center>inside the console.</center>

<center>Check your Python version, 3.6 is the one used here.<center/>

In [ ]:
%%bash
python3 -V

### <center>To clone the repository</center>
<br>
<center>we need <b>Git</b>. Run</center>
<br>
<center><code>sudo apt install git</code></center>
<br>
<center>And than ...</center>

In [ ]:
%%bash
git clone https://github.com/fprotopapa/hls4ml-tutorial.git

### <center>Set up the virtual enviroment</center>
<br>
<center>by running</center>

In [ ]:
%cd hls4ml-tutorial

In [ ]:
%%bash
python3 -m venv venv_hls4ml

In [ ]:
%%bash
source venv_hls4ml/bin/activate

<center>This will keep our system-wide Python installation clean, while we setup HLS4ML.</center>
<br>
<center>Let's update pip, the packet managment system.</center>

In [ ]:
!pip install --upgrade pip

### <center>To follow along with Jupyter notebook</center>

In [ ]:
!pip install notebook

<center>enable the virtual environment inside Jupyter</center>

In [ ]:
%%bash
ipython kernel install --user --name=.venv_hls4ml

<center>and start a session in your console</center>

<center><code>jupyter notebook</code></center>

<center>The commands above are usually executed inside the console and not in a notebook.</center>
<br>
<center>But now we have everything set up and are ready to continue</center>

## <center>Installing HLS4ML</center>

<center>Check your environment</center>

In [ ]:
!which python

<center>Python should be located inside the venv folder</center>
<br>
<center>If not check your selected kernel under the Kernel tab on the Jupyter toolbar.</center>

<center>Finally ...</center>

In [ ]:
!pip install hls4ml[profiling]

## <center><b>That's it. Now it's time to create a model from scratch!</b></center>

# <center>3. Keras to HLS4ML example</center>
## <center>Building a Keras model</center>


## <center>Creating a keras model for jet tagging</center>
<br>
<center>A lot of inspiration for this example is taken from
<br><br>
https://github.com/fastmachinelearning/hls4ml-tutorial
<br><br>
It's definitily worth to have a look at.
<br><br>
They also provide examples and information for more advanced techniques.</center>

<center>First of all, let's get some more dependecies ...</center>

In [ ]:
!pip install sklearn pydot graphviz

<center>With the additional packages installed we import some libraries needed.</center>

In [ ]:
from tensorflow.keras.utils import to_categorical
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
import numpy as np
from utils import plotting
import matplotlib.pyplot as plt
%matplotlib inline
from sklearn.metrics import accuracy_score

seed = 0
np.random.seed(seed)
import tensorflow as tf
tf.random.set_seed(seed)

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Activation, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l1
from utils.callbacks import all_callbacks

import os

<center>Create a folder to store the dataset.<center/>

In [ ]:
if os.path.exists('data'):
    print('Folder \'data\' already created.')
else:
    print('Folder \'data\' created.')
    os.mkdir('data')

<center><b>Fetch the dataset from OpenML.</b></center>
<br>
<center>Identify jets of particles from the LHC, created for the study of ultra low latency inference with hls4ml.<br>
Use 16 high level features to identify the 5 jet classes: quark (q), gluon (g), W boson (w), Z boson (z), 
or top quark (t).</center>
<br>
<center>More information is available under https://www.openml.org/d/42468</center>

In [ ]:
data = fetch_openml('hls4ml_lhc_jets_hlf')
X, y = data['data'], data['target']

<center>Now let's have a look at the data.</center>

In [ ]:
print(data['feature_names'])
print(X.shape, y.shape)
print(X[:5])
print(y[:5])

<center>Let's run some preprocessing. And save the dataset to the disk.</center>

In [ ]:
le = LabelEncoder()
y = le.fit_transform(y)
y = to_categorical(y, 5)
X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_val = scaler.fit_transform(X_train_val)
X_test = scaler.transform(X_test)

In [ ]:
np.save(os.path.join('data','X_train_val.npy'), X_train_val)
np.save(os.path.join('data','X_test.npy'), X_test)
np.save(os.path.join('data','y_train_val.npy'), y_train_val)
np.save(os.path.join('data','y_test.npy'), y_test)
np.save(os.path.join('data','classes.npy'), le.classes_)

## <center>Defining the model</center>
<br>
<center>Now it's time to create our network description.<br> Consisting out of three dense (fully connected) layer with 64, 32 and 32 neurons. <br>This network uses the ReLU activation in between and a softmax for the output layer.</center>

In [ ]:
model = Sequential()
model.add(Dense(64, input_shape=(16,), name='fc1', kernel_initializer='lecun_uniform', kernel_regularizer=l1(0.0001)))
model.add(Activation(activation='relu', name='relu1'))
model.add(Dense(32, name='fc2', kernel_initializer='lecun_uniform', kernel_regularizer=l1(0.0001)))
model.add(Activation(activation='relu', name='relu2'))
model.add(Dense(32, name='fc3', kernel_initializer='lecun_uniform', kernel_regularizer=l1(0.0001)))
model.add(Activation(activation='relu', name='relu3'))
model.add(Dense(5, name='output', kernel_initializer='lecun_uniform', kernel_regularizer=l1(0.0001)))
model.add(Activation(activation='softmax', name='softmax'))

<center>To get a nicely looking summary, keras models have a build-in method.</center>

In [ ]:
model.summary()

<center>We'll train the over 30 epoches and save it in format h5 and json.</center>

In [ ]:
train = True
if train:
    adam = Adam(lr=0.0001)
    model.compile(optimizer=adam, loss=['categorical_crossentropy'], metrics=['accuracy'])
    callbacks = all_callbacks(stop_patience = 1000,
                              lr_factor = 0.5,
                              lr_patience = 10,
                              lr_epsilon = 0.000001,
                              lr_cooldown = 2,
                              lr_minimum = 0.0000001,
                              outputDir = 'jet_tagging_keras')
    model.fit(X_train_val, y_train_val, batch_size=1024,
              epochs=30, validation_split=0.25, shuffle=True,
              callbacks = callbacks.callbacks)

    model_json = model.to_json()
    with open("jet_tagging_keras/jet_tagging.json", "w") as json_file:
        json_file.write(model_json)
else:
    from tensorflow.keras.models import load_model
    model = load_model('jet_tagging_keras/KERAS_check_best_model.h5')

<center>We did it! Our model is ready to deploy. 
A quick check of the precision is needed, though.</center>

In [ ]:
y_keras = model.predict(X_test)

In [ ]:
print("Accuracy: {}".format(accuracy_score(np.argmax(y_test, axis=1), np.argmax(y_keras, axis=1))))
plt.figure(figsize=(9,9))
_ = plotting.makeRoc(y_test, y_keras, le.classes_)

## <center>Convert to HLS</center>
<br>
<center>Again, some packages to load ...</center>

In [ ]:
import hls4ml
import yaml

<center>Add the vivado path as an environmental variable.<center/>

In [ ]:
os.environ['PATH'] = '/tools/Xilinx/Vivado/2019.2/bin:' + os.environ['PATH']

<center>To use HLS4ML to it's full extend, the hls design can be adjusted.
<br>
This can be done by writing code or using a yaml-file.</center>
<br>
<center>Here the configuraton is set within the python code.
<br>
A seperatly written file can be loaded for e.g. with</center>
<br>

<center>  
<code> with open("keras-config.yml", 'r') as ymlfile:        
    config = yaml.load(ymlfile, Loader=yaml.FullLoader)
</code>
</center>

<br>
<center>Let's read the configuration from the loaded model.</center>

In [ ]:
config = hls4ml.utils.config_from_keras_model(model, granularity='name')
print("-----------------------------------")
plotting.print_dict(config)
print("-----------------------------------")

<center><b>As shown in the network structure and the layer settings, HLS4ML alows to adjust every layer seperatly.</b></center>
<br>
<center>
Now is the time to create the hls representation and run some profiling.
</center><br>

<center>
But ...<br>
There is an import error in the profiling file which won't allow us to use get_ymodel_keras method later on.
<br>
So we need to replace the file with a modified version. (May be obsolete in the future)
</center>

In [ ]:
path_to_fix = os.path.join(os.getcwd(),'venv_hls4ml','lib','python3.6','site-packages','hls4ml','model')

os.rename(os.path.join(path_to_fix,'profiling.py'),os.path.join(path_to_fix,'profiling2.py'))
import shutil
shutil.copy(os.path.join(os.getcwd(),'profiling_wo_qkeras.py'), os.path.join(path_to_fix,'profiling.py'))

In [ ]:
from hls4ml.model import profiling
hls_model_std = hls4ml.converters.convert_from_keras_model(model,
                                                       hls_config=config,
                                                       output_dir='model_std/hls4ml_prj',
                                                       fpga_part='xc7a100tcsg324-1')

In [ ]:
hls4ml.utils.plot_model(hls_model_std, show_shapes=True, show_precision=True, to_file=None)

In [ ]:
profiling.numerical(keras_model=model, hls_model=hls_model_std, X=X_test[:1000])

<center>A good explanation of the charts can be found in the tutorial linked above. The following explanation is an excerpt of it.</center><br>
<center>
<cite>
The first thing to try is to numerically profile your model. This method plots the distribution of the weights (and biases) as a box and whisker plot.<br> The grey boxes show the values which can be represented with the data types used in the hls_model. Generally, you need the box to overlap completely with the whisker 'to the right' (large values) otherwise you'll get saturation & wrap-around issues. It can be okay for the box not to overlap completely 'to the left' (small values), but finding how small you can go is a matter of trial-and-error.
We get an overview of the network structure and the weight distribution.
</cite></center>

<center>Now let's compare the performance of the hls model with the keras model. Therefore we need to compile the hls model and run the predictions</center>

In [ ]:
hls_model_std.compile()
y_hls_std = hls_model_std.predict(np.ascontiguousarray(X_test))

In [ ]:
print("Keras  Accuracy: {}".format(accuracy_score(np.argmax(y_test, axis=1), np.argmax(y_keras, axis=1))))
print("hls4ml Accuracy: {}".format(accuracy_score(np.argmax(y_test, axis=1), np.argmax(y_hls_std, axis=1))))

fig, ax = plt.subplots(figsize=(9, 9))
_ = plotting.makeRoc(y_test, y_keras, le.classes_)
plt.gca().set_prop_cycle(None) # reset the colors
_ = plotting.makeRoc(y_test, y_hls_std, le.classes_, linestyle='--')

from matplotlib.lines import Line2D
lines = [Line2D([0], [0], ls='-'),
         Line2D([0], [0], ls='--')]
from matplotlib.legend import Legend
leg = Legend(ax, lines, labels=['keras', 'hls4ml'],
            loc='lower right', frameon=False)
ax.add_artist(leg)

<center>Now let's change the fixed-point size of the first layer and repeat the profiling.</center>

In [ ]:
config['LayerName']['fc1']['Precision']['weight'] = 'ap_fixed<8,2>'
hls_model = hls4ml.converters.convert_from_keras_model(model,
                                                       hls_config=config,
                                                       output_dir='model_1/hls4ml_prj',
                                                       fpga_part='xc7a100tcsg324-1')

In [ ]:
hls4ml.utils.plot_model(hls_model, show_shapes=True, show_precision=True, to_file=None)

In [ ]:
profiling.numerical(keras_model=model, hls_model=hls_model)

<center><b>To analyse the internal model behavior we can use the trace method.</b></center>
<br>
<center>First we change the entries to make trace usable and then we convert the hls model a second time.</center>

In [ ]:
for layer in config['LayerName'].keys():
    config['LayerName'][layer]['Trace'] = True
hls_model = hls4ml.converters.convert_from_keras_model(model,
                                                       hls_config=config,
                                                       output_dir='model_1/hls4ml_prj',
                                                       fpga_part='xc7a100tcsg324-1')

<center>After the compilation we're ready to run the predictions.</center>

In [ ]:
hls_model.compile()
hls4ml_pred, hls4ml_trace = hls_model.trace(np.ascontiguousarray(X_test[:1000]))
y_hls = hls_model.predict(np.ascontiguousarray(X_test))

<center>To be able to compare the results with those of the keras model, we need to do the same here as well.</center>

In [ ]:
keras_trace = hls4ml.model.profiling.get_ymodel_keras(model,np.ascontiguousarray( X_test[:1000]))

<center>Now let's have a look at the outputs of the first layer ...</center>

In [ ]:
print("Keras layer 'fc1', first sample:")
print(keras_trace['fc1'][0])
print("hls4ml layer 'fc1', first sample:")
print(hls4ml_trace['fc1'][0])

<center>Let's visualize the difference in performance due to the reduced weight size.</center>

In [ ]:
print("Keras  Accuracy: {}".format(accuracy_score(np.argmax(y_test, axis=1), np.argmax(y_keras, axis=1))))
print("hls4ml Accuracy: {}".format(accuracy_score(np.argmax(y_test, axis=1), np.argmax(y_hls, axis=1))))

fig, ax = plt.subplots(figsize=(9, 9))
_ = plotting.makeRoc(y_test, y_keras, le.classes_)
plt.gca().set_prop_cycle(None) # reset the colors
_ = plotting.makeRoc(y_test, y_hls, le.classes_, linestyle='--')

from matplotlib.lines import Line2D
lines = [Line2D([0], [0], ls='-'),
         Line2D([0], [0], ls='--')]
from matplotlib.legend import Legend
leg = Legend(ax, lines, labels=['keras', 'hls4ml'],
            loc='lower right', frameon=False)
ax.add_artist(leg)

## <center>4. Compilation & build process</center>

<center>For the sake of simplicity, let's delete the traced model and create a new one. In this model we'll change the model precision to a (8,2). And than build the unchanged hls model and this newly created one with vivado.</center>

In [ ]:
config = hls4ml.utils.config_from_keras_model(model, granularity='Model')
                                              
config['Model']['Precision'] = 'ap_fixed<8,2>'

hls_model = hls4ml.converters.convert_from_keras_model(model,
                                                       hls_config=config,
                                                       output_dir='model_1/hls4ml_prj',
                                                       fpga_part='xc7a100tcsg324-1')

In [ ]:
hls_model.compile()
y_hls = hls_model.predict(np.ascontiguousarray(X_test))

<center>The build processes can take some time, to monitor the logs run the commands below inside a new console:<br><br>
    <code>tail -f model_std/hls4ml_prj/vivado_hls.log</code>
<br><br>
    or
<br><br>
    <code>tail -f model_1/hls4ml_prj/vivado_hls.log</code>
</center>

In [ ]:
hls_model.build()

In [ ]:
hls_model_std.build()

## <center>5. Result of Design Flow</center>

<center>Let's have a look at the reports. <br>
First with the reduced weight size.</center>

In [ ]:
hls4ml.report.read_vivado_report('model_1/hls4ml_prj/')

<center>Than the unchanged model:</center>

In [ ]:
hls4ml.report.read_vivado_report('model_std/hls4ml_prj/')

<center>Let's have a final comparison between prediction precision.</center>

In [ ]:
print("Keras           Accuracy: {}".format(accuracy_score(np.argmax(y_test, axis=1), np.argmax(y_keras, axis=1))))
print("model unchanged Accuracy: {}".format(accuracy_score(np.argmax(y_test, axis=1), np.argmax(y_hls_std, axis=1))))
print("model reduced   Accuracy: {}".format(accuracy_score(np.argmax(y_test, axis=1), np.argmax(y_hls, axis=1))))

fig, ax = plt.subplots(figsize=(9, 9))
_ = plotting.makeRoc(y_test, y_keras, le.classes_)
plt.gca().set_prop_cycle(None) # reset the colors
_ = plotting.makeRoc(y_test, y_hls_std, le.classes_, linestyle='--')
plt.gca().set_prop_cycle(None) # reset the colors
_ = plotting.makeRoc(y_test, y_hls, le.classes_, linestyle=':')

from matplotlib.lines import Line2D
lines = [Line2D([0], [0], ls='-'),
         Line2D([0], [0], ls='--'),
         Line2D([0], [0], ls=':')]
from matplotlib.legend import Legend
leg = Legend(ax, lines, labels=['keras', 'hls4ml_std','hls4ml_red'],
            loc='lower right', frameon=False)
ax.add_artist(leg)

## <center>6. Conclusion</center>